# 2. Gold standard

Aligns the lawyers' highlights to the sentences from `01_segmentation`, and measures agreement between the two lawyers.

**Reads:** `intermediate/sentences.pkl`, and `gold_annotations.csv` from `extract_annotations.ipynb` (one row per highlight, both annotators).
**Writes:** `intermediate/gold_sentences.csv`, one row per scored sentence with its five 0/1 labels and whether it is in the dev or test split, and `test_judgments.csv`, the held out judgments. Notebooks 03 to 06 evaluate against it.

## Setup

In [ ]:
from argmine_common import *
import difflib
from collections import defaultdict
from sklearn.metrics import cohen_kappa_score

GOLD_ANNOTATOR_COL, GOLD_FILE_COL, GOLD_TEXT_COL, GOLD_LABEL_COL = "annotator", "file", "text", "category"

# If a gold file cannot be linked to a judgment automatically, add it here: {"file name": doc_id}
FILE_TO_DOC_ID = {}

GOLD_MERGE = "union"           # two annotators on one judgment: "union" or "intersection"
ALIGN_MIN_COVER = 0.5          # share of a sentence that must fall inside a highlight
N_TEST_DOCS = 3                # annotated judgments held out for the final test

corpus = load_corpus()
TITLE = titles(corpus)
sents = load_sentences()

## Load highlights and align them to sentences

Steps:
1. Load the combined highlight file and map labels to C1..C5. Labels that cannot be mapped (for example "unconfirmed_yellow") are reported and their sentences are excluded from scoring, rather than counted as negatives.
2. Link each file to a judgment: first by matching the file name to the judgment title, then by text overlap, then via `FILE_TO_DOC_ID`.
3. Align highlights to sentences. A sentence takes a highlight's label when at least `ALIGN_MIN_COVER` of it falls inside the highlight, or when the highlight sits wholly inside the sentence.

**Assumption to confirm with the lawyers:** highlights are exhaustive, so any reasoning sentence they left unhighlighted belongs to none of the five categories. If they only marked examples, recall scores below will be underestimated.

In [ ]:
def norm_text(s):
    s = re.sub(r"(\w)-\s+(\w)", r"\1\2", str(s))
    s = re.sub(r"[^a-z0-9]+", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()

def norm_title(s):
    s = str(s)
    s = re.sub(r"\.(pdf|docx?|txt|gdoc)$", "", s, flags=re.I)
    return norm_text(s.replace("_", " "))


gold_spans = (pd.read_csv(GOLD_CSV)
              .rename(columns={GOLD_ANNOTATOR_COL: "annotator", GOLD_FILE_COL: "file",
                               GOLD_TEXT_COL: "text", GOLD_LABEL_COL: "raw_label"})
              [["annotator", "file", "raw_label", "text"]]
              .dropna(subset=["text"])
              .reset_index(drop=True))
gold_spans["annotator"] = gold_spans.annotator.str.lower()
gold_spans["label"] = gold_spans.raw_label.map(normalize_label)
gold_spans["span_id"] = range(len(gold_spans))

print(f"{len(gold_spans)} highlights from {gold_spans.file.nunique()} files")
print(pd.crosstab(gold_spans.raw_label.astype(str), gold_spans.label.fillna("UNMAPPED")).to_string())

In [ ]:
# ── Link files to judgments ───────────────────────────────────────────────────
title_index = {norm_title(t): d for d, t in TITLE.items()}

def shingles(s, n=6):
    toks = norm_text(s).split()
    return {" ".join(toks[i:i + n]) for i in range(max(len(toks) - n + 1, 0))}

shingle_index = defaultdict(set)
link_source = "full_text" if "full_text" in corpus.columns else "reason_text"
for d, t in zip(corpus.doc_id, corpus[link_source].fillna("")):
    for sh in shingles(t):
        shingle_index[sh].add(d)

def link_by_content(texts, min_share=0.3):
    hits, total = defaultdict(int), 0
    for t in texts:
        for sh in shingles(t):
            total += 1
            for d in shingle_index.get(sh, ()):
                hits[d] += 1
    if not hits or total == 0:
        return None, 0.0
    d, h = max(hits.items(), key=lambda x: x[1])
    return (d, h / total) if h / total >= min_share else (None, h / total)

links = []
for f, grp in gold_spans.groupby("file"):
    if f in FILE_TO_DOC_ID:
        links.append((f, FILE_TO_DOC_ID[f], "manual", 1.0)); continue
    d = title_index.get(norm_title(f))
    if d is None:
        close = difflib.get_close_matches(norm_title(f), list(title_index), n=1, cutoff=0.9)
        d = title_index[close[0]] if close else None
    if d is not None:
        links.append((f, d, "title", 1.0)); continue
    d, share = link_by_content(grp.text.tolist())
    links.append((f, d, "content" if d is not None else "UNLINKED", round(share, 2)))

link_df = pd.DataFrame(links, columns=["file", "doc_id", "how", "text_match"])
print(link_df.to_string(index=False))
gold_spans = gold_spans.merge(link_df[["file", "doc_id"]], on="file", how="left")

In [ ]:
# ── Align highlights to sentences ─────────────────────────────────────────────
def coverage(sent_n, span_n):
    if not sent_n:
        return 0.0
    if sent_n in span_n:
        return 1.0
    sm = difflib.SequenceMatcher(None, sent_n, span_n, autojunk=False)
    return sum(b.size for b in sm.get_matching_blocks() if b.size >= 12) / len(sent_n)

sents["norm"] = sents.text.map(norm_text)
align_rows = []
for _, sp in gold_spans.dropna(subset=["doc_id"]).iterrows():
    span_n = norm_text(sp.text)
    span_tok = set(span_n.split())
    cand = sents[sents.doc_id == sp.doc_id]
    for sid, sent_n in zip(cand.index, cand.norm):
        tok = sent_n.split()
        if not tok or sum(t in span_tok for t in tok) / len(tok) < 0.3:
            continue
        cov = coverage(sent_n, span_n)
        inside = len(span_n.split()) >= 6 and span_n in sent_n
        if cov >= ALIGN_MIN_COVER or inside:
            align_rows.append({"sid": sid, "span_id": sp.span_id, "annotator": sp.annotator,
                               "label": sp.label, "cover": round(cov, 2)})
aligned = pd.DataFrame(align_rows, columns=["sid", "span_id", "annotator", "label", "cover"])

span_report = gold_spans.assign(n_sents=gold_spans.span_id.map(aligned.span_id.value_counts()).fillna(0).astype(int))
print(f"Highlights aligned to at least one sentence: {(span_report.n_sents > 0).mean():.0%}")
print(span_report.groupby("annotator").apply(lambda g: (g.n_sents > 0).mean()).round(2).to_string())

# Unaligned highlights usually lie outside the court_reasoning and conclusion sections.
unaligned = span_report[(span_report.n_sents == 0) & span_report.doc_id.notna()]
if len(unaligned):
    full = dict(zip(corpus.doc_id, corpus.get("full_text", corpus.reason_text).fillna("").map(norm_text)))
    unaligned = unaligned.assign(in_full_text=[norm_text(t)[:80] in full.get(d, "")
                                               for t, d in zip(unaligned.text, unaligned.doc_id)])
    print(f"\n{len(unaligned)} unaligned highlights; found elsewhere in the full judgment: "
          f"{unaligned.in_full_text.sum()}")
    display(unaligned[["annotator", "file", "label", "in_full_text", "text"]].head(10))

In [ ]:
# ── Gold label matrix over all sentences of the annotated judgments ───────────
gold_docs = sorted(aligned.merge(sents[["doc_id"]], left_on="sid", right_index=True).doc_id.unique())
G = sents[sents.doc_id.isin(gold_docs)].copy()

ann_by_doc = gold_spans.dropna(subset=["doc_id"]).groupby("doc_id").annotator.unique().map(set)
lab = aligned.dropna(subset=["label"]).groupby(["sid", "label"]).annotator.unique().map(set)

def gold_value(sid, cat, doc):
    who = lab.get((sid, cat), set())
    if GOLD_MERGE == "intersection" and len(ann_by_doc.get(doc, ())) > 1:
        return int(who >= ann_by_doc[doc])
    return int(len(who) > 0)

Y = np.array([[gold_value(s, c, d) for c in CATS] for s, d in zip(G.index, G.doc_id)])
unscored = set(aligned[aligned.label.isna()].sid)          # sentences under unmapped labels
keep = ~G.index.isin(unscored)
G, Y = G[keep], Y[keep]
DOCS = G.doc_id.values
GOLD_IDX = G.index.values          # row positions into `sents`

print(f"Gold set: {len(gold_docs)} judgments, {len(G)} sentences, {len(unscored)} excluded (unmapped labels)")
print(pd.DataFrame({"positive sentences": Y.sum(0),
                    "judgments with category": pd.DataFrame(Y).groupby(DOCS).max().sum().values},
                   index=[SHORT[c] for c in CATS]).to_string())

## Hold out test judgments

`N_TEST_DOCS` annotated judgments are set aside as a test set. Whole judgments are held out, not sentences, because sentences of one judgment are alike and would leak between the splits. Among 500 random draws we keep the one whose judgments cover the most categories, so the test set is not missing a category by chance.

The draw is saved to `argument_mining/test_judgments.csv` and reused on every later run, so adding annotations does not reshuffle it. Delete that file to draw again, but only before anyone has looked at test scores.

In [ ]:
if TEST_SPLIT_CSV.exists():
    TEST_DOCS = set(pd.read_csv(TEST_SPLIT_CSV).doc_id)
    gone = TEST_DOCS - set(gold_docs)
    if gone:
        print(f"Warning: test judgments no longer in the gold set: {sorted(gone)}. "
              f"Delete {TEST_SPLIT_CSV.name} to draw again.")
else:
    assert len(gold_docs) >= N_TEST_DOCS + 3, f"Only {len(gold_docs)} gold judgments: too few to hold out {N_TEST_DOCS}"
    doc_cats = pd.DataFrame(Y, columns=CATS).groupby(DOCS).max()
    rng = np.random.default_rng(SEED)
    draws = [rng.choice(doc_cats.index, N_TEST_DOCS, replace=False) for _ in range(500)]
    TEST_DOCS = set(max(draws, key=lambda d: doc_cats.loc[d].max().sum()))
    (pd.DataFrame({"doc_id": sorted(TEST_DOCS)}).assign(title=lambda t: t.doc_id.map(TITLE))
     .to_csv(TEST_SPLIT_CSV, index=False))
    print(f"Drew {len(TEST_DOCS)} test judgments and saved them to {TEST_SPLIT_CSV.relative_to(BASE)}")

TEST = np.isin(DOCS, list(TEST_DOCS))
for name, mask in (("dev", ~TEST), ("test", TEST)):
    print(f"\n{name}: {len(set(DOCS[mask]))} judgments, {mask.sum()} sentences")
    print(pd.Series(Y[mask].sum(0), index=[SHORT[c] for c in CATS], name="positive sentences").to_string())
for d in sorted(TEST_DOCS):
    print("test:", TITLE.get(d, d))

## Save

In [ ]:
gold_table = (pd.DataFrame(Y, columns=CATS)
              .assign(sid=GOLD_IDX, doc_id=DOCS, split=np.where(TEST, "test", "dev"))
              [["sid", "doc_id", "split", *CATS]])
gold_table.to_csv(GOLD_SENTS_CSV, index=False)
print(f"Saved {len(gold_table)} gold sentences to {GOLD_SENTS_CSV.relative_to(BASE)}")

## Agreement between the two lawyers

In [ ]:
# ── Agreement between the two lawyers (judgments both annotated) ──────────────
shared = [d for d, who in ann_by_doc.items() if len(who) > 1 and d in gold_docs]
if shared:
    rows_s = np.isin(DOCS, shared)
    sids_s = GOLD_IDX[rows_s]
    annots = sorted(set.union(*[ann_by_doc[d] for d in shared]))[:2]
    kap = {}
    for c in CATS:
        a = [int(annots[0] in lab.get((s, c), set())) for s in sids_s]
        b = [int(annots[1] in lab.get((s, c), set())) for s in sids_s]
        kap[SHORT[c]] = round(cohen_kappa_score(a, b), 2) if (sum(a) + sum(b)) else np.nan
    print(f"Cohen's kappa, {annots[0]} vs {annots[1]}, {len(shared)} shared judgments, {rows_s.sum()} sentences")
    print(pd.Series(kap).to_string())
else:
    print("No judgment was annotated by both lawyers, so agreement cannot be computed yet. "
          "Double annotating 5 or more judgments would let us report it.")